# Fase 4 — Extração de Informações e Grafo de Conhecimento

**Objetivo**: Extrair entidades, padrões e relações do corpus Wikipedia (1100 artigos) construindo um grafo de conhecimento com análise de centralidade.

**Corpus**: 1100 artigos Wikipedia formatados (saída das Fases 1 e 2)

**Tecnologias**:
- `spaCy >= 3.7` — NER com modelo `pt_core_news_lg`
- `python-Levenshtein` — Normalização de entidades
- `NetworkX >= 3.1` — Grafo de conhecimento e centralidade
- `matplotlib`, `seaborn` — Visualizações

**Pergunta analítica**: *Quais entidades são mais centrais no corpus e por quê?*

---

In [ ]:
# Setup: imports, constantes inline, criar diretórios
import os
import sys
import logging
from IPython.display import Image, display

# Adiciona src ao path
SRC_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from extractor import TextExtractor
from normalizer import EntityNormalizer
from graph_builder import KnowledgeGraphBuilder

# Caminhos inline
BASE_DIR    = os.path.dirname(os.path.abspath('__file__'))
INPUT_DIR   = os.path.join(BASE_DIR, 'input')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'output')
PLOTS_DIR   = os.path.join(OUTPUT_DIR, 'plots')

PARQUET_PATH   = os.path.join(INPUT_DIR, '1100-artigos_wikipedia-formatados-v001_lemmatizacao.parquet')
ARTEFATO_PATH  = os.path.join(INPUT_DIR, 'fase2_artifact.lpf2')

for d in [INPUT_DIR, OUTPUT_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Logging básico
logging.basicConfig(level=logging.WARNING)
print('Setup concluído.')
print(f'  src:    {SRC_DIR}')
print(f'  output: {OUTPUT_DIR}')

## Etapa 1 — Carregamento do Corpus

In [ ]:
import pandas as pd

# Tenta artefato Fase 2; fallback: reconstrói do parquet Fase 1
documentos = None
titulos = None

if os.path.exists(ARTEFATO_PATH):
    try:
        import joblib
        artefato = joblib.load(ARTEFATO_PATH)
        documentos = artefato.documentos
        titulos    = artefato.titulos
        print(f'Artefato Fase 2 carregado: {len(documentos)} documentos')
    except Exception as e:
        print(f'Aviso: falha ao carregar artefato ({e}), usando parquet.')

if documentos is None:
    parquet_df = pd.read_parquet(PARQUET_PATH)
    docs, titles = [], []
    for art_id in sorted(parquet_df['id_artigo'].unique()):
        sub = parquet_df[parquet_df['id_artigo'] == art_id]
        col = next((c for c in ('processado', 'lema', 'token') if c in sub.columns), 'token')
        docs.append(' '.join(sub[col].dropna().astype(str)))
        titles.append(
            sub['titulo'].iloc[0] if 'titulo' in sub.columns else f'Doc {art_id}'
        )
    documentos, titulos = docs, titles
    print(f'Parquet Fase 1 carregado: {len(documentos)} documentos')

print(f'\nExemplo (doc 0, primeiros 200 chars):')
print(documentos[0][:200])

## Etapa 2 — Extração de Entidades (NER)

In [ ]:
extractor = TextExtractor()

print('Executando NER com spaCy...')
df_entities = extractor.extract_entities_from_docs(documentos)

print(f'Total de entidades extraídas: {len(df_entities)}')
print(f'\nDistribuição por tipo:')
print(df_entities['label'].value_counts().to_string())

In [ ]:
import matplotlib.pyplot as plt

# Gráfico inline: distribuição por tipo de entidade
tipo_counts = df_entities['label'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
tipo_counts.plot(kind='bar', ax=ax, color='#4C72B0')
ax.set_title('Distribuição de entidades por tipo (NER spaCy)', fontsize=13)
ax.set_xlabel('Tipo de entidade')
ax.set_ylabel('Frequência')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Etapa 3 — Extração de Padrões (Regex)

In [ ]:
print('Extraindo padrões com regex...')
df_patterns = extractor.extract_patterns_from_docs(documentos, titulos)

print(f'Total de ocorrências: {len(df_patterns)}')
if not df_patterns.empty:
    print('\nOcorrências por tipo de padrão:')
    print(df_patterns['tipo'].value_counts().to_string())
    print('\nExemplos:')
    display(df_patterns.groupby('tipo').head(2)[['tipo', 'valor']].reset_index(drop=True))
else:
    print('Nenhum padrão encontrado no corpus.')

## Etapa 4 — Extração de Relações (SVO)

In [ ]:
print('Extraindo relações SVO...')
df_relations = extractor.extract_relations_from_docs(documentos)

stats = extractor.get_statistics()
print(f'Relações extraídas: {stats["relacoes"]}')

if not df_relations.empty:
    print('\nExemplos de triplas SVO:')
    display(
        df_relations[df_relations['tipo'] == 'svo'][['sujeito', 'verbo', 'objeto']]
        .drop_duplicates()
        .head(10)
        .reset_index(drop=True)
    )
else:
    print('Nenhuma relação SVO encontrada.')

## Etapa 5 — Normalização de Entidades (Levenshtein)

In [ ]:
normalizer = EntityNormalizer(threshold=2)

print('Normalizando entidades por tipo...')
df_normalized = normalizer.normalize_by_type(df_entities)

# Estatísticas de redução
if not df_entities.empty:
    entidades_unicas = df_entities['text'].dropna().unique().tolist()
    grupos = normalizer.normalize_entities(entidades_unicas)
    mapa   = normalizer.get_canonical_mapping(grupos)
    s = normalizer.get_normalization_stats(entidades_unicas, grupos)
    print(f'\nEntidades originais : {s["total_original"]}')
    print(f'Formas canônicas    : {s["total_canonical"]}')
    print(f'Taxa de redução     : {s["reduction_rate"]*100:.1f}%')
    print(f'Tamanho médio grupo : {s["avg_group_size"]}')

    # Exporta mapeamento
    normalizer.export_mapping(mapa, os.path.join(OUTPUT_DIR, 'entidades_normalizadas.csv'))
    print(f'\nMapeamento salvo em output/entidades_normalizadas.csv')

## Etapa 6 — Construção do Grafo de Conhecimento

In [ ]:
graph = KnowledgeGraphBuilder()

print('Construindo grafo a partir das relações e entidades...')
graph.build_from_dataframes(df_relations, df_normalized)

stats_g = graph.get_graph_stats()
print(f'\nEstatísticas do grafo:')
for k, v in stats_g.items():
    print(f'  {k:15s}: {v}')

## Etapa 7 — Análise de Centralidade

In [ ]:
print('Calculando métricas de centralidade...')
centralidade = graph.calculate_centrality()

print(f'Métricas calculadas: {list(centralidade.keys())}')

# Ranking top-10 por betweenness
top10 = graph.get_top_nodes('betweenness', top_n=10)
if top10:
    print('\nTop-10 entidades por betweenness centrality:')
    for rank, (entidade, valor) in enumerate(top10, 1):
        grau = graph.graph.degree(entidade)
        print(f'  {rank:2d}. {entidade:<30s} betweenness={valor:.4f}  grau={grau}')
else:
    print('Dados insuficientes para calcular centralidade.')

## Etapa 8 — Visualizações

In [ ]:
print('Gerando os 4 gráficos...')
graph.generate_all_plots(df_normalized, PLOTS_DIR)
print('Gráficos salvos em output/plots/')

In [ ]:
# Exibição inline dos 4 gráficos
import os
from IPython.display import Image, display

graficos = [
    ('01_knowledge_graph.png',       'Grafo de Conhecimento'),
    ('02_centrality_distribution.png', 'Distribuição de Centralidade'),
    ('03_top_entities_by_type.png',  'Top Entidades por Tipo'),
    ('04_relations_heatmap.png',     'Heatmap de Co-ocorrência'),
]

for filename, titulo in graficos:
    caminho = os.path.join(PLOTS_DIR, filename)
    if os.path.exists(caminho):
        print(f'\n### {titulo}')
        display(Image(filename=caminho, width=900))
    else:
        print(f'Gráfico não encontrado: {filename}')

## Etapa 9 — Síntese Interpretativa

### Pergunta Analítica

> **Quais entidades são mais centrais no corpus e por quê?**

A *betweenness centrality* mede a frequência com que um nó aparece nos caminhos mais curtos entre outros nós.  
Entidades com alta betweenness **conectam clusters temáticos distintos**, aparecendo em múltiplos contextos — funcionam como *hubs* da narrativa do corpus.

A análise é complementada por outras três métricas:
- **Degree centrality** — volume de conexões diretas
- **Closeness centrality** — distância média para todos os outros nós
- **Eigenvector centrality** — importância dos vizinhos (como PageRank)

In [ ]:
import json

# Exportar CSV e JSON final
extractor.export_to_csv(df_entities, df_patterns, df_relations, OUTPUT_DIR)
graph.export_csv(
    os.path.join(OUTPUT_DIR, 'nos_grafo.csv'),
    os.path.join(OUTPUT_DIR, 'relacoes_extraidas.csv'),
)

stats_extracao = extractor.get_statistics()
resultados = {
    'corpus':  {'documentos': len(documentos)},
    'extracao': stats_extracao,
    'grafo':   graph.get_graph_stats(),
    'top_entidades': [
        {'entidade': n, 'betweenness': round(v, 6)}
        for n, v in graph.get_top_nodes('betweenness', 10)
    ],
}

caminho_json = os.path.join(OUTPUT_DIR, 'resultados.json')
with open(caminho_json, 'w', encoding='utf-8') as fh:
    json.dump(resultados, fh, ensure_ascii=False, indent=2, default=str)

print('Outputs exportados:')
for f in os.listdir(OUTPUT_DIR):
    if os.path.isfile(os.path.join(OUTPUT_DIR, f)):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
        print(f'  {f:<45s} {size:>8d} bytes')

## Conclusão

### Resposta à Pergunta Analítica

As entidades mais centrais no corpus são aquelas com maior **betweenness centrality** — i.e., as que aparecem como intermediárias nos caminhos que ligam diferentes partes do grafo.

Isso indica que essas entidades são mencionadas em múltiplos contextos temáticos distintos, conectando artigos sobre pessoas, organizações e locais que de outra forma seriam clusters isolados.

Em corpora enciclopédicos (como Wikipedia), os *hubs* costumam ser:
- **Países e capitais** (alta co-ocorrência com entidades de múltiplos domínios)
- **Organizações internacionais** (ONU, UEFA, etc.)
- **Datas históricas** (conectam eventos de diferentes áreas)

### Reprodutibilidade

Para reproduzir esta análise do zero:
```bash
# 1. Instalar dependências
pip install -r requirements.txt
python -m spacy download pt_core_news_lg

# 2. Garantir inputs em fase4/input/
#    - 1100-artigos_wikipedia-formatados-v001_lemmatizacao.parquet
#    - fase2_artifact.lpf2  (opcional)

# 3. Executar o notebook célula a célula
#    OU via script:
python fase4/src/main.py
```

**Todos os outputs são determinísticos** (seed=42 nos layouts de grafo).